# GPU Capacity Risk Assessment Agent on Amazon Bedrock

**Disclaimer:** This is sample code, for non-production usage; work with your security and legal teams to meet your organizational security, regulatory, and compliance requirements before deployment.

This notebook demonstrates how to build an AI agent that performs **capacity risk assessment** for GPU inference workloads on Amazon SageMaker AI and Amazon EC2 using **Amazon Bedrock** with the Converse API and tool use. The agent is **model-agnostic** — the default is Anthropic Claude Haiku 4.5 (fastest & cheapest current-generation Anthropic model on Bedrock, $1/M input, $5/M output), with per-request escalation to Sonnet 4.5 or Nova Pro — any Bedrock model supporting the Converse API with `tool_use` (Anthropic Claude, Amazon Nova, Meta Llama, Mistral, OpenAI on Bedrock) works by changing one string.

## What you'll learn
- How to use the Bedrock Converse API for tool-use orchestration (provider-neutral)
- Building a multi-tool agent loop with bounded iterations
- Confidence scoring based on 5 signals (availability, pricing, quota, fleet size, region maturity)
- Capacity strategy recommendations (ODCR, Capacity Blocks, phased deployment)
- How data-residency and government / regulatory constraints (GovCloud, FedRAMP, ITAR, GDPR, LGPD) narrow region and model choices

## Problem
ML platform teams and TAMs spend hours assessing GPU capacity risk — checking availability, pricing, quotas, and figuring out whether a deployment will actually land. Each exercise touches 3-4 disconnected data sources.

## Solution
An agent that accepts a natural-language capacity ask and returns a risk-scored decision package in seconds.

```
Input:  "22 g6 instances in af-south-1 for prod inference launching in 2 weeks"
Output:
  Confidence: VERY LOW (<0.3)
  Verdict:    NOT FEASIBLE (current state)
  Why:        g6 not GA for SageMaker in af-south-1
  Action:     Switch to g5 family (GA) or change region
```

## Prerequisites

- AWS account with Amazon Bedrock access
- Model access enabled for at least one Converse-API-capable model (default here: Anthropic Claude Haiku 4.5)
- AWS credentials configured (`aws configure` or environment variables)
- Python 3.10+

For **regulated deployments** (GovCloud, FedRAMP, ITAR, GDPR, LGPD), see the "Model Selection Guide" section of `docs/implementation-guide.md` before running this notebook — you'll want to override `BEDROCK_MODEL_ID` and `AWS_REGION` to values inside your compliance boundary.

In [ ]:
# Install dependencies
!pip install -q boto3

In [ ]:
import json
import boto3
from functools import lru_cache

# Configuration — default is Claude Haiku 4.5 via US cross-region inference
# profile (fastest & cheapest current-generation Anthropic model on Bedrock,
# $1/M input, $5/M output). Any Bedrock model supporting Converse API with
# tool_use works. See docs/implementation-guide.md "Model Selection Guide"
# for alternatives (stronger reasoning tiers, GovCloud-eligible models, etc.).
BEDROCK_MODEL_ID = "us.anthropic.claude-haiku-4-5-20251001-v1:0"
AWS_REGION = "us-west-2"

print(f"Model: {BEDROCK_MODEL_ID}")
print(f"Region: {AWS_REGION}")

## Step 1: Define Tools

Each tool wraps a public AWS API. The agent decides which tools to call based on the user's ask.

### Tool 1: Instance Availability Check
Uses `ec2.DescribeInstanceTypeOfferings` to determine if an instance type is offered in a region.

In [ ]:
@lru_cache(maxsize=8)
def _ec2_client(region):
    return boto3.client("ec2", region_name=region)

def check_instance_availability(instance_type: str, region: str) -> dict:
    """Check if an instance type is offered in a region and which AZs support it."""
    bare_type = instance_type.removeprefix("ml.")
    try:
        client = _ec2_client(region)
        resp = client.describe_instance_type_offerings(
            LocationType="region",
            Filters=[{"Name": "instance-type", "Values": [bare_type]}],
        )
        offered = len(resp.get("InstanceTypeOfferings", [])) > 0
        if not offered:
            return {
                "instance_type": bare_type, "region": region,
                "available": False, "azs": [],
                "interpretation": f"{bare_type} is NOT offered in {region}.",
                "source": "ec2.DescribeInstanceTypeOfferings",
            }
        az_resp = client.describe_instance_type_offerings(
            LocationType="availability-zone",
            Filters=[{"Name": "instance-type", "Values": [bare_type]}],
        )
        azs = sorted(o["Location"] for o in az_resp.get("InstanceTypeOfferings", []))
        return {
            "instance_type": bare_type, "region": region,
            "available": True, "azs": azs, "az_count": len(azs),
            "interpretation": f"{bare_type} offered in {region} across {len(azs)} AZ(s): {', '.join(azs)}.",
            "source": "ec2.DescribeInstanceTypeOfferings",
        }
    except Exception as exc:
        return {"instance_type": bare_type, "region": region, "available": None,
                "interpretation": f"API error: {exc}", "source": "ec2.DescribeInstanceTypeOfferings"}

# Quick test
print(json.dumps(check_instance_availability("g5.2xlarge", "us-east-1"), indent=2))

### Tool 2: SageMaker Pricing
Uses the AWS Pricing API to get on-demand hourly rates.

In [ ]:
REGION_LOCATION = {
    "us-east-1": "US East (N. Virginia)", "us-east-2": "US East (Ohio)",
    "us-west-2": "US West (Oregon)", "eu-central-1": "EU (Frankfurt)",
    "ap-south-1": "Asia Pacific (Mumbai)", "sa-east-1": "South America (Sao Paulo)",
}

@lru_cache(maxsize=1)
def _pricing_client():
    return boto3.client("pricing", region_name="us-east-1")

def get_sagemaker_pricing(instance_type: str, region: str) -> dict:
    """Get SageMaker hosting on-demand hourly price."""
    ml_type = instance_type if instance_type.startswith("ml.") else f"ml.{instance_type}"
    location = REGION_LOCATION.get(region)
    if not location:
        return {"instance_type": ml_type, "region": region, "hourly_usd": None,
                "available": False, "source": "AWS Pricing API",
                "_notes": f"Unknown region: {region}"}
    try:
        resp = _pricing_client().get_products(
            ServiceCode="AmazonSageMaker",
            Filters=[
                {"Type": "TERM_MATCH", "Field": "instanceName", "Value": ml_type},
                {"Type": "TERM_MATCH", "Field": "location", "Value": location},
                {"Type": "TERM_MATCH", "Field": "platoinstancetype", "Value": "Hosting"},
            ],
            MaxResults=10,
        )
        for entry_str in resp.get("PriceList", []):
            entry = json.loads(entry_str) if isinstance(entry_str, str) else entry_str
            for _, term in entry.get("terms", {}).get("OnDemand", {}).items():
                for _, dim in term.get("priceDimensions", {}).items():
                    rate = dim.get("pricePerUnit", {}).get("USD")
                    if rate:
                        return {"instance_type": ml_type, "region": region,
                                "hourly_usd": float(rate), "available": True,
                                "source": "AWS Pricing API"}
        return {"instance_type": ml_type, "region": region, "hourly_usd": None,
                "available": False, "source": "AWS Pricing API",
                "_notes": f"No pricing found — likely not GA in {region}."}
    except Exception as exc:
        return {"instance_type": ml_type, "region": region, "hourly_usd": None,
                "available": False, "source": "AWS Pricing API", "_notes": str(exc)}

# Quick test
print(json.dumps(get_sagemaker_pricing("g5.2xlarge", "us-east-1"), indent=2))

### Tool 3: Service Quotas
Checks current SageMaker endpoint instance limits.

In [ ]:
def get_sagemaker_quota(instance_type: str, region: str) -> dict:
    """Get SageMaker endpoint quota for an instance type."""
    ml_type = instance_type if instance_type.startswith("ml.") else f"ml.{instance_type}"
    try:
        client = boto3.client("service-quotas", region_name=region)
        paginator = client.get_paginator("list_service_quotas")
        for page in paginator.paginate(ServiceCode="sagemaker"):
            for q in page.get("Quotas", []):
                if ml_type in q.get("QuotaName", "").lower() and "endpoint" in q.get("QuotaName", "").lower():
                    return {"instance_type": ml_type, "region": region, "found": True,
                            "quota_name": q["QuotaName"], "current_limit": q["Value"],
                            "adjustable": q.get("Adjustable", False),
                            "source": "service-quotas.ListServiceQuotas"}
        return {"instance_type": ml_type, "region": region, "found": False,
                "interpretation": f"No endpoint quota found for {ml_type} in {region}.",
                "source": "service-quotas.ListServiceQuotas"}
    except Exception as exc:
        return {"instance_type": ml_type, "region": region, "found": False,
                "interpretation": f"API error: {exc}", "source": "service-quotas.ListServiceQuotas"}

### Tool 4: Alternative Instance Finder
Finds similar GPU instances available in the target region.

In [ ]:
GPU_FAMILIES = ["g5", "g6", "g6e", "g7", "g7e", "p4d", "p5", "p5en", "p6-b200", "p6-b300", "inf2", "trn1"]

def find_alternative_instances(instance_type: str, region: str, max_results: int = 5) -> dict:
    """Find alternative GPU instances ranked by similarity."""
    bare_type = instance_type.removeprefix("ml.")
    client = _ec2_client(region)
    try:
        ref = client.describe_instance_types(InstanceTypes=[bare_type])
        if not ref.get("InstanceTypes"):
            ref = boto3.client("ec2", region_name="us-east-1").describe_instance_types(InstanceTypes=[bare_type])
        rt = ref["InstanceTypes"][0]
        rg = rt.get("GpuInfo", {}).get("Gpus", [{}])[0]
        rs = {"vcpus": rt["VCpuInfo"]["DefaultVCpus"], "mem": rt["MemoryInfo"]["SizeInMiB"]/1024,
              "gpus": rg.get("Count",0), "gpu_mem": (rg.get("MemoryInfo",{}).get("SizeInMiB",0) or 0)/1024}
        offered = set()
        for page in client.get_paginator("describe_instance_type_offerings").paginate(LocationType="region"):
            for o in page["InstanceTypeOfferings"]:
                if o["InstanceType"].split(".")[0] in GPU_FAMILIES:
                    offered.add(o["InstanceType"])
        offered.discard(bare_type)
        candidates = []
        for ct in sorted(offered)[:30]:
            try:
                cr = client.describe_instance_types(InstanceTypes=[ct])["InstanceTypes"][0]
                cg = cr.get("GpuInfo",{}).get("Gpus",[{}])[0]
                cs = {"vcpus": cr["VCpuInfo"]["DefaultVCpus"], "mem": cr["MemoryInfo"]["SizeInMiB"]/1024,
                      "gpus": cg.get("Count",0), "gpu_mem": (cg.get("MemoryInfo",{}).get("SizeInMiB",0) or 0)/1024}
                score = sum([0.3*min(rs["gpus"],cs["gpus"])/max(rs["gpus"],cs["gpus"],1),
                             0.3*min(rs["gpu_mem"],cs["gpu_mem"])/max(rs["gpu_mem"],cs["gpu_mem"],1),
                             0.2*min(rs["vcpus"],cs["vcpus"])/max(rs["vcpus"],cs["vcpus"],1),
                             0.2*min(rs["mem"],cs["mem"])/max(rs["mem"],cs["mem"],1)])
                candidates.append({"instance_type": ct, "similarity": round(score,2)})
            except Exception: continue
        candidates.sort(key=lambda x: x["similarity"], reverse=True)
        return {"reference": bare_type, "region": region, "alternatives": candidates[:max_results],
                "source": "ec2.DescribeInstanceTypes"}
    except Exception as exc:
        return {"reference": bare_type, "region": region, "alternatives": [], "source": str(exc)}

## Step 2: Build the Agent

The agent uses **Bedrock Converse API** with `toolConfig`. The loop:
1. Send message + tool definitions → model
2. Model returns `tool_use` → execute → feed results back
3. Model returns `end_turn` → done
4. Bounded by `max_iterations`

In [ ]:
SYSTEM_PROMPT = """You are a GPU Capacity Planning Agent that assesses the RISK of deploying GPU inference workloads on Amazon SageMaker. You think like a senior capacity planner — not just checking data, but scoring deployment confidence.

When given a capacity request, you:
1. Check instance availability in the target region
2. Look up SageMaker hosting pricing (confirms GA status)
3. Check current SageMaker endpoint quotas
4. If blocked, find alternative GPU instances
5. Assess overall deployment RISK based on all signals

Structure your response as:

## Capacity Risk Assessment

### Confidence Score
- **HIGH (0.8-1.0)**: Available, quota sufficient, established region, small fleet
- **MODERATE (0.5-0.7)**: Available but quota needs increase, or moderate fleet
- **MODERATE-LOW (0.3-0.5)**: Large fleet, limited region, or mixed signals
- **VERY LOW (<0.3)**: Quota far below ask, limited region, large fleet, or multiple risk factors

### Signals Table
- Instance offered in region: yes/no (source: DescribeInstanceTypeOfferings)
- Pricing available (GA status): yes/no (source: AWS Pricing API)
- Quota vs. requested: current limit vs. ask (source: Service Quotas)
- Fleet size risk: small <10 | moderate 10-50 | large 50-100 | very large 100+
- Region maturity: established | newer | limited-capacity

### Verdict
- **DEPLOYABLE** — All signals green, proceed
- **DEPLOYABLE WITH PREPARATION** — Needs quota increase or ODCR
- **HIGH RISK** — Technically possible but significant capacity risk
- **NOT FEASIBLE (current state)** — Cannot be fulfilled without capacity assistance

## GPU Chip Reference (always include)
- g5 → NVIDIA A10G (24 GB)  |  g6 → NVIDIA L4 (24 GB)  |  g6e → NVIDIA L40S (48 GB)
- g7 → NVIDIA RTX PRO 4500 Blackwell (32 GB)  |  g7e → NVIDIA RTX PRO 6000 Blackwell (96 GB)
- p4d → NVIDIA A100 (40/80 GB)  |  p5 → NVIDIA H100 (80 GB, incl. single-GPU p5.4xlarge)  |  p5e → NVIDIA H200 (141 GB)
- p6-b200 → NVIDIA B200 Blackwell (192 GB)  |  p6-b300 → NVIDIA B300 Blackwell Ultra (288 GB)
- inf2 → AWS Inferentia2  |  trn1/trn2 → AWS Trainium

## Region Maturity Context
- Established: us-east-1, us-west-2, eu-west-1, ap-northeast-1
- Moderate: eu-central-1, ap-southeast-1, ap-southeast-2, ca-central-1
- Limited-capacity: af-south-1, sa-east-1, ap-south-2, me-south-1

## Recommendations (based on risk level)
- MODERATE-LOW or lower: ODCR, Capacity Blocks, phased deployment (25%→50%→100%)
- VERY LOW / NOT FEASIBLE: AWS Support case, account team engagement, alternative regions/families
- Always include: numbered next steps, timeline awareness, escalation path for large fleets
- Always end with: \"Full SageMaker AI pricing: https://aws.amazon.com/sagemaker/pricing/\"

Rules:
- NEVER fabricate pricing, availability, or quota data
- If a tool call fails, report the failure transparently
- Always include GPU chip names when listing instance types
- For large fleets (>50) in limited regions, always flag capacity risk
- Never promise capacity — you provide signals, not guarantees
- Distinguish 'not offered' (categorical) vs 'quota too low' (fixable)"""

TOOLS = {
    "check_instance_availability": {
        "handler": check_instance_availability,
        "spec": {"toolSpec": {"name": "check_instance_availability",
            "description": "Check if instance type is offered in a region and which AZs support it.",
            "inputSchema": {"json": {"type":"object","properties":{
                "instance_type":{"type":"string"},"region":{"type":"string"}},"required":["instance_type","region"]}}}},
    },
    "get_sagemaker_pricing": {
        "handler": get_sagemaker_pricing,
        "spec": {"toolSpec": {"name": "get_sagemaker_pricing",
            "description": "Get SageMaker hosting on-demand hourly price. None if not GA.",
            "inputSchema": {"json": {"type":"object","properties":{
                "instance_type":{"type":"string"},"region":{"type":"string"}},"required":["instance_type","region"]}}}},
    },
    "get_sagemaker_quota": {
        "handler": get_sagemaker_quota,
        "spec": {"toolSpec": {"name": "get_sagemaker_quota",
            "description": "Get SageMaker endpoint instance quota limit for a region.",
            "inputSchema": {"json": {"type":"object","properties":{
                "instance_type":{"type":"string"},"region":{"type":"string"}},"required":["instance_type","region"]}}}},
    },
    "find_alternative_instances": {
        "handler": find_alternative_instances,
        "spec": {"toolSpec": {"name": "find_alternative_instances",
            "description": "Find alternative GPU instances in a region ranked by similarity.",
            "inputSchema": {"json": {"type":"object","properties":{
                "instance_type":{"type":"string"},"region":{"type":"string"},
                "max_results":{"type":"integer"}},"required":["instance_type","region"]}}}},
    },
}
print(f"Registered {len(TOOLS)} tools: {', '.join(TOOLS.keys())}")

In [ ]:
def run_agent(user_message: str, max_iterations: int = 8, verbose: bool = True) -> dict:
    """Run the capacity planning agent."""
    client = boto3.client("bedrock-runtime", region_name=AWS_REGION)
    messages = [{"role": "user", "content": [{"text": user_message}]}]
    tool_config = {"tools": [t["spec"] for t in TOOLS.values()]}
    tool_calls, total_in, total_out = [], 0, 0

    for i in range(1, max_iterations + 1):
        if verbose: print(f"\n--- Iteration {i} ---")
        resp = client.converse(modelId=BEDROCK_MODEL_ID, messages=messages,
            toolConfig=tool_config, system=[{"text": SYSTEM_PROMPT}],
            inferenceConfig={"maxTokens": 4096, "temperature": 0.0})
        usage = resp.get("usage", {})
        total_in += usage.get("inputTokens", 0)
        total_out += usage.get("outputTokens", 0)
        msg = resp["output"]["message"]
        messages.append(msg)
        stop = resp.get("stopReason")
        if verbose: print(f"  Stop: {stop} | +{usage.get('inputTokens',0)}in +{usage.get('outputTokens',0)}out")
        if stop == "end_turn": break
        if stop == "tool_use":
            results = []
            for block in msg.get("content", []):
                if "toolUse" not in block: continue
                tu = block["toolUse"]
                name, args = tu["name"], tu.get("input", {})
                if verbose: print(f"  -> {name}({json.dumps(args, separators=(',',':'))})")
                result = TOOLS[name]["handler"](**args)
                tool_calls.append((name, args, result))
                results.append({"toolResult":{"toolUseId":tu["toolUseId"],"content":[{"json":result}]}})
            messages.append({"role":"user","content":results})
            continue
        break
    final = "\n".join(b["text"] for b in messages[-1].get("content",[]) if "text" in b)
    return {"answer": final, "tool_calls": tool_calls, "iterations": i,
            "usage": {"input_tokens": total_in, "output_tokens": total_out}}

## Step 3: Run Demo Scenarios

### Scenario A: Simple Feasibility — Low Risk
Small model, established region. Should show HIGH confidence.

In [ ]:
result = run_agent("I need to deploy a Mistral 7B model for real-time inference in us-west-2. What's my risk?")
print("\n" + "="*60 + "\nAGENT ANSWER:\n" + "="*60)
print(result["answer"])
print(f"\nStats: {result['iterations']} iters, {len(result['tool_calls'])} tools, "
      f"{result['usage']['input_tokens']}+{result['usage']['output_tokens']} tokens")

### Scenario B: Moderate Risk — New Region, Tight Timeline
22 g6 instances in Cape Town with a 2-week deadline. Should show VERY LOW / NOT FEASIBLE.

In [ ]:
result = run_agent("I'm planning 22 g6 instances in af-south-1 for prod inference launching in 2 weeks. What's my risk?")
print("\n" + "="*60 + "\nAGENT ANSWER:\n" + "="*60)
print(result["answer"])
print(f"\nStats: {result['iterations']} iters, {len(result['tool_calls'])} tools, "
      f"{result['usage']['input_tokens']}+{result['usage']['output_tokens']} tokens")

### Scenario C: Adversarial — Large Fleet, Data Residency Constraint
300 GPU instances in São Paulo with LGPD data residency preventing cross-region pivot.

In [ ]:
result = run_agent(
    "A large enterprise customer needs 175 ml.g5.2xlarge + 100 ml.g5.4xlarge + 25 multi-GPU instances "
    "in sa-east-1 for production AI expansion. Data residency prevents cross-region pivot. "
    "What are the options?"
)
print("\n" + "="*60 + "\nAGENT ANSWER:\n" + "="*60)
print(result["answer"])
print(f"\nStats: {result['iterations']} iters, {len(result['tool_calls'])} tools, "
      f"{result['usage']['input_tokens']}+{result['usage']['output_tokens']} tokens")

## Step 4: Cost Analysis

In [ ]:
# Bedrock model rates vary widely. Set IN_RATE/OUT_RATE from
# https://aws.amazon.com/bedrock/pricing/ for the specific BEDROCK_MODEL_ID
# you're using. The illustrative values below match Claude Haiku 4.5
# ($1/M input, $5/M output) — the guidance's default model. Update these
# if you swap to a different model (e.g. Sonnet 4.5 at $3/$15 per M).
IN_RATE = 0.001 / 1000
OUT_RATE = 0.005 / 1000

print(f"{'Scenario':<40} {'In':>7} {'Out':>7} {'Iters':>6} {'Cost':>8}")
print("-"*72)
for name, query in [
    ("A: Mistral 7B us-west-2", "Deploy Mistral 7B in us-west-2. Risk?"),
    ("B: 22 g6 af-south-1", "22 g6 instances in af-south-1 for prod in 2 weeks. Risk?"),
    ("C: 300 instances sa-east-1", "175 g5.2xlarge + 100 g5.4xlarge + 25 multi-GPU in sa-east-1. Options?"),
]:
    r = run_agent(query, verbose=False)
    cost = r['usage']['input_tokens']*IN_RATE + r['usage']['output_tokens']*OUT_RATE
    print(f"{name:<40} {r['usage']['input_tokens']:>7,} {r['usage']['output_tokens']:>7,} {r['iterations']:>6} ${cost:>6.4f}")

## Key Takeaways

1. **Capacity risk assessment, not just feasibility** — The agent scores deployment confidence and recommends capacity strategies, not just yes/no answers
2. **Confidence scoring based on 5 signals** — Availability, pricing (GA status), quota vs. ask, fleet size, and region maturity combine into a single confidence score
3. **Region maturity awareness** — The agent knows af-south-1 and sa-east-1 are limited-capacity regions with higher ICE risk for large fleets
4. **Escalation recommendations** — For high-stakes decisions (>50 instances, limited regions), the agent recommends engaging AWS account teams directly rather than trying to solve alone
5. **GPU chip names in all output** — Every instance recommendation includes the GPU chip (e.g., "ml.g6.2xlarge (NVIDIA L4 – 24 GB)") for clarity
6. **Cost per run**: ~$0.10-0.30 for complex multi-instance queries — orders of magnitude cheaper than hours of manual research

## Next Steps
- Deploy with `sam deploy` — uses a Function URL for complex queries (no 29s API Gateway timeout)
- **Production features**: WAF, retry with exponential backoff, response cache, CloudWatch alarms, token budget guardrails
- **Customize the system prompt** for your customer context (add specific discount tiers, preferred regions, compliance requirements)
- Run `examples/demo_recorded.py` for an offline, deterministic walkthrough of the pre-built demo scenarios